In [9]:
import numpy as np
import nmslib
import pickle
import time
import os
import psutil
from tqdm import tqdm

# ─── Load from cache ──────────────────────────────────────────────────────────
qt_10k        = np.load('/tmp/qt_10k.npy')
qtree_full    = np.load('/tmp/qtree_vectors_full.npy')
with open('/tmp/gt_lookup_10k.pkl',  'rb') as f: gt_10k  = pickle.load(f)
with open('/tmp/gt_lookup_full.pkl', 'rb') as f: gt_full = pickle.load(f)

QUERY_START_10K = 8000
QUERY_START_ID  = 187019
THREADS         = 32   # fixed for all notebooks — apples to apples

corpus_10k  = qt_10k[:QUERY_START_10K]
query_10k   = qt_10k[QUERY_START_10K:]
corpus_full = qtree_full[:QUERY_START_ID]
query_full  = qtree_full[QUERY_START_ID:]

print(f"10k:  corpus={corpus_10k.shape} | queries={query_10k.shape}")
print(f"Full: corpus={corpus_full.shape} | queries={query_full.shape}")
print(f"10k GT: {len(gt_10k)} | Full GT: {len(gt_full)}")

# ─── Utilities ────────────────────────────────────────────────────────────────
def recall_at_k(gt_lookup, nbrs, query_start_id, K):
    total = 0.0; count = 0
    for i, (ids, _) in enumerate(nbrs):
        qid = query_start_id + i
        gt  = set(gt_lookup.get(qid, [])[:K])
        if not gt: continue
        total += len(gt & set(ids[:K])) / len(gt)
        count += 1
    return total / count if count > 0 else 0.0

def eval_recall_all(gt_lookup, nbrs, query_start_id):
    return {k: recall_at_k(gt_lookup, nbrs, query_start_id, k)
            for k in [10, 50, 100, 500]}

def get_mem_mb():
    return psutil.Process(os.getpid()).memory_info().rss / 1024**2

def run_baseline(corpus, query, gt_lookup, query_start_id, label):
    print(f"\n{'='*60}")
    print(f"BASELINE — {label}")
    print(f"{'='*60}")

    vec_mb = corpus.nbytes / 1024**2
    print(f"Vector size: {vec_mb:.1f} MB | dim={corpus.shape[1]}")

    # Build index
    m0  = get_mem_mb()
    idx = nmslib.init(method='hnsw', space='WeightedJaccard')
    for i in tqdm(range(len(corpus)), desc="Adding", mininterval=2.0):
        idx.addDataPoint(i, corpus[i])
    t0 = time.time()
    idx.createIndex({'M': 20, 'efConstruction': 200, 'post': 1},
                    print_progress=True)
    build_s  = time.time() - t0
    idx_mb   = get_mem_mb() - m0
    print(f"Build: {build_s:.1f}s | Index mem: {idx_mb:.1f} MB")

    # Query at K=500 with 32 threads
    idx.setQueryTimeParams({'efSearch': 200})
    t0   = time.time()
    nbrs = idx.knnQueryBatch(query, k=500, num_threads=THREADS)
    qps  = len(query) / (time.time() - t0)
    print(f"QPS ({THREADS} threads): {qps:.1f}")

    rec = eval_recall_all(gt_lookup, nbrs, query_start_id)
    for k, r in rec.items():
        print(f"  R@{k:<4} = {r:.4f}")

    return {
        'label': label,
        'dim': corpus.shape[1],
        'vec_mb': vec_mb,
        'idx_mb': idx_mb,
        'build_s': build_s,
        'qps': qps,
        **{f'R@{k}': v for k, v in rec.items()}
    }

# ─── Run both scales ──────────────────────────────────────────────────────────
res_10k  = run_baseline(corpus_10k,  query_10k,  gt_10k,
                         QUERY_START_10K, "10k dataset")
res_full = run_baseline(corpus_full, query_full, gt_full,
                         QUERY_START_ID,  "Full 233k dataset")

# ─── Final summary table ──────────────────────────────────────────────────────
print(f"\n{'='*75}")
print(f"BASELINE FINAL SUMMARY — HNSW WeightedJaccard ({THREADS} threads)")
print(f"{'='*75}")
print(f"{'Metric':<25} {'10k':>20} {'Full (233k)':>20}")
print(f"-"*65)
rows = [
    ('R@10',          'R@10'),
    ('R@50',          'R@50'),
    ('R@100',         'R@100'),
    ('R@500',         'R@500'),
    ('qps',           'QPS'),
    ('build_s',       'Build time (s)'),
    ('vec_mb',        'Vector size (MB)'),
    ('idx_mb',        'Index mem (MB)'),
    ('dim',           'Vector dim'),
]
for key, label in rows:
    v10 = res_10k[key]
    vf  = res_full[key]
    print(f"  {label:<23} {v10:>20.2f} {vf:>20.2f}")

# Save results for comparison in other notebooks
import pickle
with open('/tmp/results_baseline.pkl', 'wb') as f:
    pickle.dump({'10k': res_10k, 'full': res_full}, f)
print(f"\nResults saved to /tmp/results_baseline.pkl")

10k:  corpus=(8000, 18499) | queries=(2000, 18499)
Full: corpus=(187019, 18220) | queries=(46754, 18220)
10k GT: 1818 | Full GT: 44666

BASELINE — 10k dataset
Vector size: 564.5 MB | dim=18499


Adding: 100%|██████████| 8000/8000 [00:00<00:00, 83984.76it/s]

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

Build: 51.7s | Index mem: 20.3 MB
QPS (32 threads): 246.4
  R@10   = 0.9966
  R@50   = 0.9986
  R@100  = 0.9990
  R@500  = 0.9974

BASELINE — Full 233k dataset
Vector size: 12998.5 MB | dim=18220


Adding: 100%|██████████| 187019/187019 [00:04<00:00, 45977.01it/s]

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
*

Build: 559.7s | Index mem: 11519.4 MB
QPS (32 threads): 610.8
  R@10   = 0.9925
  R@50   = 0.9953
  R@100  = 0.9963
  R@500  = 0.9864

BASELINE FINAL SUMMARY — HNSW WeightedJaccard (32 threads)
Metric                                     10k          Full (233k)
-----------------------------------------------------------------
  R@10                                    1.00                 0.99
  R@50                                    1.00                 1.00
  R@100                                   1.00                 1.00
  R@500                                   1.00                 0.99
  QPS                                   246.45               610.80
  Build time (s)                         51.66               559.71
  Vector size (MB)                      564.54             12998.53
  Index mem (MB)                         20.27             11519.41
  Vector dim                          18499.00             18220.00

Results saved to /tmp/results_baseline.pkl
